In [1]:
import numpy as np
import time
from numba import cuda

In [11]:
@cuda.jit
def binary_search_kernel(arr, targets, results):

    idx = cuda.grid(1)

    if idx < targets.size:

        target = targets[idx]

        left = 0
        right = arr.size - 1

        found = -1

        while left <= right:

            mid = (left + right) // 2

            if arr[mid] == target:
                found = mid
                break

            elif arr[mid] < target:
                left = mid + 1

            else:
                right = mid - 1

        results[idx] = found

In [12]:
def cpu_binary_search(arr, target):

    left = 0
    right = len(arr) - 1

    while left <= right:

        mid = (left + right) // 2

        if arr[mid] == target:
            return mid

        elif arr[mid] < target:
            left = mid + 1

        else:
            right = mid - 1

    return -1

In [13]:
N = 10_000_000

sorted_array = np.arange(N, dtype=np.int32)

# Increase workload for GPU
NUM_SEARCHES = 5_000_000

targets = np.random.randint(
    0,
    N,
    size=NUM_SEARCHES,
    dtype=np.int32
)

print("Dataset Size :", N)
print("Search Queries :", NUM_SEARCHES)

Dataset Size : 10000000
Search Queries : 5000000


In [14]:
start_cpu = time.time()

cpu_results = np.array(
    [cpu_binary_search(sorted_array, x)
     for x in targets],
    dtype=np.int32
)

cpu_time = time.time() - start_cpu


In [15]:
gpu_total_start = time.time()

d_arr = cuda.to_device(sorted_array)
d_targets = cuda.to_device(targets)
d_results = cuda.device_array(NUM_SEARCHES,
                              dtype=np.int32)

threads_per_block = 256

blocks_per_grid = (
    NUM_SEARCHES +
    threads_per_block - 1
) // threads_per_block


In [16]:
kernel_start = time.time()

binary_search_kernel[
    blocks_per_grid,
    threads_per_block
](
    d_arr,
    d_targets,
    d_results
)

cuda.synchronize()

kernel_time = time.time() - kernel_start

# Download Results
gpu_results = d_results.copy_to_host()

gpu_total_time = time.time() - gpu_total_start

In [17]:
correct = np.array_equal(
    cpu_results,
    gpu_results
)


In [18]:
print("\n===== Verification =====")
print("Results Match :", correct)

print("\n===== Performance =====")
print(f"CPU Time               : {cpu_time:.4f} sec")
print(f"GPU Kernel Time Only   : {kernel_time:.4f} sec")
print(f"GPU Total Time         : {gpu_total_time:.4f} sec")

print("\n===== Speedup =====")

print(
    f"Kernel Speedup = "
    f"{cpu_time/kernel_time:.2f}x"
)

print(
    f"Overall Speedup = "
    f"{cpu_time/gpu_total_time:.2f}x"
)


===== Verification =====
Results Match : True

===== Performance =====
CPU Time               : 40.7768 sec
GPU Kernel Time Only   : 0.1248 sec
GPU Total Time         : 11.9099 sec

===== Speedup =====
Kernel Speedup = 326.81x
Overall Speedup = 3.42x
